# 02 — Extraction des frappes de dunun (V2)

On transforme les pistes audio de dunun en un **tableau exploitable** : une ligne par frappe, avec sa **position dans le cycle** (subdivision 1–12 + position relative précise) et ses features de timbre.

> Note importante (corrigée) : le dunun se joue avec **un seul type de frappe** — ce n'est pas comme le djembé (claqué/tonique/bass). Ce qui change d'un dunun à l'autre, c'est sa **taille** (donc sa hauteur) : c'est la distinction **D1 / D2**. On ne classe donc PAS les frappes de dunun par type ; on étudie *où* et *quand* il frappe.

Sortie : `output/dunun_frappes_V2.csv`.

## 0. Imports

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from IPython.display import display
import utils_V2 as u
d = u.list_dunun(); print(len(d), 'pistes de dunun')

## 1. Aligner chaque frappe sur le cycle + extraire les features

In [ ]:
rows = []
for _, r in d.iterrows():
    y = u.load_track(r['piece'], r['take'], r['code'])
    cp = u.cycle_positions(r['piece'], r['take'], r['code'])   # SD, rel, dur, start, time
    if cp.empty:
        continue
    f = u.fast_stroke_features(y, cp['time'].values).merge(cp[['time','SD','rel','dur']], on='time', how='left')
    f['piece'], f['take'], f['code'] = r['piece'], r['take'], r['code']
    rows.append(f)
df = pd.concat(rows, ignore_index=True)
print(len(df), 'frappes de dunun alignées sur le cycle')
display(df.head())

## 2. Aperçu — répartition par rythme et par dunun (D1/D2)

In [ ]:
display(df.groupby(['piece','code']).size().unstack(fill_value=0))
print('D1 = dunun principal, D2 = 2e dunun (couverture partielle selon les prises)')
display(u.dunun_coverage()[['piece','take','D1_wav','D2_wav','D1_selected','D2_selected','paired']])

## 3. Sauvegarde

In [ ]:
df.to_csv('output/dunun_frappes_V2.csv', index=False)
print('Sauvegardé -> output/dunun_frappes_V2.csv |', len(df), 'frappes')
print('colonnes :', list(df.columns))

## Bilan
- Chaque frappe de dunun est **datée et positionnée dans le cycle** (subdivision + position relative).
- Pas de typage de frappe (le dunun n'en a qu'un) ; la variable pertinente est **quel dunun** (D1/D2) et **son motif dans le temps**.
- **Suite `03_dunun_comparaison_V2`** : comparer les 3 rythmes (motif, densité, micro-timing, tempo, D1 vs D2).